# Analyse Att — v3 (minimal)

Une seule chose, rien d'autre :

1. Lire chaque tag Att minute par minute depuis OIA (référence `tag_`).
2. Nettoyer : ne garder que des pas entiers dans `[1, 32]`, tout le reste (null, non entier, hors plage) est remplacé par la dernière valeur de pas valide connue.
3. Réduire en plateaux : une table par tag `pas`, `debut`, `duree_min`.
4. Ajouter un numéro d'opération qui incrémente à chaque retour au pas 1 (nouveau cycle).

Pas de table de référence process, pas de notion de défaut. Une table par tag : `pas`, `debut`, `duree_min`, `operation`.


In [22]:
import json
import sys

import numpy as np
import pandas as pd

from tools.OI_class_OP import OI_DataProcessor
from tools.oi_cache import save, load, restore_into

pd.set_option('display.max_columns', None)

for _mod in list(sys.modules):
    if _mod == 'att_tags_config' or _mod.startswith('att_tags_config.'):
        del sys.modules[_mod]

from att_tags_config import TAGS_ as tags, is_batch

batch_tags = [t for t in tags if is_batch(t)]

processor = OI_DataProcessor(
    url_base='https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    start='2022-01-01',
    end='2026-07-31',
    tags_selected=tags,
    tags_other=[],
    interval='PT01M',
    verbose=False,
    agg='MEAN',
)

CACHE_PATH = f"documents/confidentiel/cache/att_{processor.start}_{processor.end}.pkl.gz"
cached = load(CACHE_PATH)
if cached is not None:
    restore_into(processor, cached)
    print(f"Données chargées depuis le cache : {CACHE_PATH}")
else:
    processor.merge()
    cache_file = save(processor, CACHE_PATH)
    print(f"Données récupérées depuis OIA et mises en cache : {cache_file} ({cache_file.stat().st_size / 1e6:.1f} Mo)")

print(f"{len(batch_tags)} tags batch sur {len(tags)}")


Données chargées depuis le cache : documents/confidentiel/cache/att_2022-01-01_2026-07-31.pkl.gz
21 tags batch sur 28


## 1. Nettoyage : ne garder que des pas entiers valides

Le tag `tag_` donne directement le numéro de pas (pas de `repère // 100` à
faire). Bruit attendu : null, valeurs non entières (moyennage `agg=MEAN`
lors d'une transition de pas dans la minute) et valeurs hors `[1, 32]`
(dont 0, qui ne correspond à aucun pas du séquenceur). Tout ça est remplacé
par la dernière valeur de pas valide connue (`ffill`).


In [23]:
PAS_MIN, PAS_MAX = 1, 32


def clean_pas(series: pd.Series, pas_min: int = PAS_MIN, pas_max: int = PAS_MAX) -> pd.Series:
    """Ne garde que les pas entiers dans [pas_min, pas_max] ; le reste (null, non entier, hors plage) est remplacé par la dernière valeur valide."""
    is_integer = series.notna() & np.isclose(series, series.round())
    is_valid = is_integer & series.between(pas_min, pas_max)
    return series.where(is_valid).ffill().astype('Int64')


pas_cols = [t['nom'] for t in batch_tags if t['nom'] in processor.data.columns]
pas = processor.data[pas_cols].apply(clean_pas)

n_dropped = pas.isna().sum()
print(f"{len(pas_cols)} tags batch nettoyés")
print(n_dropped[n_dropped > 0].rename('valeurs sans pas valide antérieur (début de série)'))


21 tags batch nettoyés
Series([], Name: valeurs sans pas valide antérieur (début de série), dtype: int64)


## 2. Table par opé : pas, début, durée

Réduction en plateaux (`nop_analysis.extract_operations`, déjà utilisé pour
ce même modèle "valeur constante par palier" ailleurs dans le projet) :
une ligne par changement de pas, avec le temps passé dans ce pas.


In [24]:
from nop_analysis import extract_operations


def build_pas_table(tag_name: str, pas_series: pd.Series) -> pd.DataFrame:
    """Table de plateaux (pas, debut, duree_min) pour un tag."""
    df = pas_series.dropna().astype('int64').to_frame('pas')
    steps = extract_operations(df, value_col='pas')
    steps = steps.rename(columns={'value': 'pas', 'operation_id': 'step_id', 'start': 'debut', 'duration_min': 'duree_min'})
    steps.insert(0, 'tag', tag_name)
    return steps[['tag', 'step_id', 'pas', 'debut', 'duree_min', 'n_samples']]


pas_tables = pd.concat([build_pas_table(c, pas[c]) for c in pas.columns], ignore_index=True)
pas_tables


,tag,step_id,pas,debut,duree_min,n_samples
0,PU1410VA,0,14,2022-01-01 00:00:00+00:00,13.0,13
1,PU1410VA,1,15,2022-01-01 00:13:00+00:00,22.0,22
2,PU1410VA,2,16,2022-01-01 00:35:00+00:00,158.0,158
3,PU1410VA,3,17,2022-01-01 03:13:00+00:00,30.0,30
4,PU1410VA,4,20,2022-01-01 03:43:00+00:00,3.0,3
...,...,...,...,...,...,...
701152,PU3320VA,53977,16,2026-07-21 17:00:00+00:00,46.0,46
701153,PU3320VA,53978,17,2026-07-21 17:46:00+00:00,2.0,2
701154,PU3320VA,53979,16,2026-07-21 17:48:00+00:00,45.0,45
701155,PU3320VA,53980,20,2026-07-21 18:33:00+00:00,17.0,17


## 3. Numéro d'opération

Incrémente à chaque retour au pas de relance du tag (nouveau cycle), par
tag. **Pas forcément le pas 1** : la quasi-totalité des tags a un
`pas_attente` documenté dans `att_operations_config.json` différent de 1
(le chaînage réel ne repasse jamais par le pas 1, cf. sa doc dans
`att_tags_config.py`) — l'utiliser comme ancre plutôt qu'un seuil global
sous-comptait fortement les cycles (et faisait disparaître du récap les
tags qui n'atteignent jamais le pas 1 sur la période). Repli sur le pas 1
si `pas_attente` est absent ou jamais observé sur la période (cas
`PU3220VA`/`PU3320VA` : le pas déclaré est trop fugace pour être capturé à
la minute).

**Garde-fou de durée minimale** : un retour sur le pas d'ancrage ne compte
comme nouvelle opération que s'il survient au moins `temps_reference_min`
(catalogue) après le précédent redémarrage. En dessous, c'est un retour
bruité *au milieu* d'un cycle réel (ex. boucle de défaut), pas un vrai
redémarrage — `temps_reference_min` est documenté comme un **minimum**
connu de l'exploitant, donc par définition aucune opération réelle ne peut
être plus courte. Un seuil à 25 % de cette référence (essayé d'abord, sur
2 mois de données) laissait passer ~2 % d'opérations sous le minimum
documenté (jusqu'à 25-49 % de la référence) une fois la période étendue à
plusieurs années — le seuil à 100 % élimine ces faux redémarrages sans
perdre de vraies opérations (~1.6 % du volume total en moins).

Les lignes avant la première occurrence acceptée de l'ancre
(`operation == 0`) sont un cycle incomplet en tête de série — à exclure
des analyses de durée.


In [25]:
MIN_CYCLE_FRACTION = 1.0  # part min. de temps_reference_min avant qu'un retour sur l'ancre compte comme nouvelle opération (100% : c'est un minimum documenté, pas une moyenne)


def resolve_anchor(pas_series: pd.Series, tag_def: dict) -> int:
    """Pas d'ancrage du cycle : `pas_attente` du catalogue, ou repli sur PAS_MIN
    si ce pas n'est jamais observé dans les données nettoyées (cas où le pas
    déclaré est trop fugace pour être capturé à la minute, ex. PU3220VA/PU3320VA)."""
    declared = tag_def.get('pas_attente') or PAS_MIN
    if (pas_series == declared).any():
        return declared
    print(f"[ATTENTION] {tag_def['nom']} : pas_attente={declared} jamais observé sur la période, repli sur pas {PAS_MIN}")
    return PAS_MIN


def assign_operations(g: pd.DataFrame, anchor: int, min_duration: float) -> pd.Series:
    """Numérote les opérations d'un tag : incrémente à chaque retour au pas
    d'ancrage, sauf si ce retour survient moins de `min_duration` minutes
    après le précédent redémarrage (retour bruité au milieu d'un cycle réel,
    absorbé dans l'opération en cours plutôt que de créer une opération
    fantôme de quelques minutes)."""
    op_id = 0
    started = False
    since_boundary = 0.0
    ops = []
    for pas_val, duree in zip(g['pas'], g['duree_min']):
        if pas_val == anchor and (not started or since_boundary >= min_duration):
            op_id += 1
            started = True
            since_boundary = 0.0
        ops.append(op_id)
        since_boundary += duree
    return pd.Series(ops, index=g.index)


anchor_pas = {t['nom']: resolve_anchor(pas[t['nom']], t) for t in batch_tags if t['nom'] in pas.columns}
temps_reference_min = {t['nom']: t.get('temps_reference_min') for t in batch_tags}

pas_tables = pas_tables.sort_values(['tag', 'debut']).reset_index(drop=True)
pas_tables['operation'] = pd.concat([
    assign_operations(g, anchor_pas[tag], (temps_reference_min.get(tag) or 0) * MIN_CYCLE_FRACTION)
    for tag, g in pas_tables.groupby('tag')
]).sort_index()
pas_tables


,tag,step_id,pas,debut,duree_min,n_samples,operation
0,PU1410VA,0,14,2022-01-01 00:00:00+00:00,13.0,13,0
1,PU1410VA,1,15,2022-01-01 00:13:00+00:00,22.0,22,0
2,PU1410VA,2,16,2022-01-01 00:35:00+00:00,158.0,158,0
3,PU1410VA,3,17,2022-01-01 03:13:00+00:00,30.0,30,0
4,PU1410VA,4,20,2022-01-01 03:43:00+00:00,3.0,3,0
...,...,...,...,...,...,...,...
701152,PU3320VA,53977,16,2026-07-21 17:00:00+00:00,46.0,46,4002
701153,PU3320VA,53978,17,2026-07-21 17:46:00+00:00,2.0,2,4002
701154,PU3320VA,53979,16,2026-07-21 17:48:00+00:00,45.0,45,4002
701155,PU3320VA,53980,20,2026-07-21 18:33:00+00:00,17.0,17,4002


## 4. Durée totale par opération

`pas_tables` liste des **plateaux de pas** (un pas peut durer 1 minute,
c'est normal) — pas des opérations. Une opération = somme des `duree_min`
de tous les pas qui la composent, du redémarrage accepté (section 3)
jusqu'au suivant. C'est cette durée-là qu'on compare à
`temps_reference_min`, pas celle d'un pas isolé.


In [26]:
operations = (
    pas_tables[pas_tables['operation'] > 0]
    .groupby(['tag', 'operation'])
    .agg(debut=('debut', 'min'), duree_min=('duree_min', 'sum'), n_pas=('pas', 'size'))
    .reset_index()
)
operations['temps_reference_min'] = operations['tag'].map(temps_reference_min)
operations['ratio_reference'] = operations['duree_min'] / operations['temps_reference_min']

print(f"Durée d'opération min observée : {operations['duree_min'].min():.0f} min (attendu : >= 100% de temps_reference_min par construction, section 3)")
operations


Durée d'opération min observée : 95 min (attendu : >= 100% de temps_reference_min par construction, section 3)


,tag,operation,debut,duree_min,n_pas,temps_reference_min,ratio_reference
0,PU1410VA,1,2022-01-01 05:05:00+00:00,893.0,15,425,2.101176
1,PU1410VA,2,2022-01-01 19:58:00+00:00,531.0,14,425,1.249412
2,PU1410VA,3,2022-01-02 04:49:00+00:00,519.0,14,425,1.221176
3,PU1410VA,4,2022-01-02 13:28:00+00:00,899.0,14,425,2.115294
4,PU1410VA,5,2022-01-03 04:27:00+00:00,661.0,16,425,1.555294
...,...,...,...,...,...,...,...
69194,PU3320VA,3998,2026-07-20 12:20:00+00:00,412.0,13,283,1.455830
69195,PU3320VA,3999,2026-07-20 19:12:00+00:00,392.0,15,283,1.385159
69196,PU3320VA,4000,2026-07-21 01:44:00+00:00,450.0,13,283,1.590106
69197,PU3320VA,4001,2026-07-21 09:14:00+00:00,461.0,13,283,1.628975


## 5. Récap par tag

Une ligne par tag : plage de pas observée, nombre d'opérations sur la
période, durées d'opération (minutes) min/médiane/max, et les opérations
liées (`documents/confidentiel/operations_network.json`, construit par
`build_operations_network.py` à partir des `operation_liee`/`sens_liaison`
de chaque `OPxxxx_pas_reference.csv`) : à quelles autres opérations celle-ci
envoie/reçoit un produit, ou attend une réponse.

Pas de colonne `ope_min`/`ope_max` : la numérotation des opérations est
contiguë par construction (section 3), donc `ope_min` vaut toujours 1 et
`ope_max` vaut toujours `nb_ope` — aucune des deux n'apporte d'info, et le
nom prête à confusion avec une durée en minutes.


In [27]:
recap = (
    pas_tables[pas_tables['operation'] > 0]
    .groupby('tag')
    .agg(pas_min=('pas', 'min'), pas_max=('pas', 'max'), nb_ope=('operation', 'nunique'))
    .reset_index()
    .merge(
        operations.groupby('tag')['duree_min'].agg(duree_minutes_min='min', duree_minutes_median='median', duree_minutes_max='max').reset_index(),
        on='tag',
    )
)
recap['temps_reference_min'] = recap['tag'].map(temps_reference_min)
recap['ratio_mediane_reference'] = recap['duree_minutes_median'] / recap['temps_reference_min']


def liaisons_for(code: str, liaisons: list) -> list:
    """Opérations liées à `code` (source ou cible), avec le sens documenté
    (ATTEND/ENVOIE/BIDIRECTIONNEL) et la direction du lien par rapport à `code`."""
    out = []
    for l in liaisons:
        if l['operation_source'] == code:
            out.append(f"->{l['operation_cible']} ({l['sens']})")
        elif l['operation_cible'] == code:
            out.append(f"<-{l['operation_source']} ({l['sens']})")
    return sorted(set(out))


with open('documents/confidentiel/operations_network.json', encoding='utf-8') as f:
    operations_network = json.load(f)

operation_code = {t['nom']: t.get('operation') for t in batch_tags}
recap['liaisons'] = recap['tag'].map(lambda t: liaisons_for(operation_code[t], operations_network['liaisons']))
recap


,tag,pas_min,pas_max,nb_ope,duree_minutes_min,duree_minutes_median,duree_minutes_max,temps_reference_min,ratio_mediane_reference,liaisons
0,PU1410VA,1,23,3086,426.0,541.0,76051.0,425,1.272941,"[->OP1420 (BIDIRECTIONNEL), ->OP1610 (BIDIRECT..."
1,PU1420VA,1,15,3096,235.0,536.0,75654.0,222,2.414414,"[->OP1410 (ATTEND), <-OP1410 (BIDIRECTIONNEL)]"
2,PU1430VA,1,28,1038,1052.0,1738.0,76770.0,1027,1.692308,"[->OP1510 (ENVOIE), <-OP1510 (ATTEND)]"
3,PU1510VA,1,30,3519,418.0,480.0,78288.0,412,1.165049,"[->OP1430 (ATTEND), ->OP1520 (ATTEND), <-OP143..."
4,PU1520VA,1,30,3479,387.0,483.0,70381.0,387,1.248062,"[->OP1510 (ATTEND), ->OP1530 (ATTEND), <-OP151..."
5,PU1530VA,1,30,3392,418.0,489.5,70458.0,418,1.171053,"[->OP1520 (ATTEND), ->OP1540 (ATTEND), <-OP152..."
6,PU1540VA,1,30,3478,391.0,486.0,70520.0,391,1.242967,"[->OP1520 (ATTEND), ->OP1610 (ENVOIE), <-OP153..."
7,PU2310VA,1,17,3050,531.0,578.0,67582.0,530,1.090566,"[->OP2320 (BIDIRECTIONNEL), <-OP2320 (BIDIRECT..."
8,PU2320VA,1,22,3097,336.0,589.0,47543.0,327,1.801223,"[->OP2310 (BIDIRECTIONNEL), ->OP2330 (ENVOIE),..."
9,PU2340VA,1,8,3119,95.0,583.0,56874.0,90,6.477778,"[->OP2320 (ATTEND), ->OP2420 (ENVOIE), <-OP232..."


Sauvegarde du récap (sans `liaisons`, déjà dans `operations_network.json`) pour
que `build_operations_graph_export.py` puisse construire le graphe frontend
(React/Next) sans stats codées en dur.


In [28]:
RECAP_PATH = "documents/confidentiel/recap_operations.csv"
recap.drop(columns=["liaisons"]).to_csv(RECAP_PATH, index=False)
print(f"Récap sauvegardé : {RECAP_PATH} ({len(recap)} tags)")


Récap sauvegardé : documents/confidentiel/recap_operations.csv (21 tags)


## 6. Diagnostic : `pas_attente` crédible ?

7 tags ont une `ratio_mediane_reference` très élevée (`PU1420VA`, `PU1430VA`,
`PU2810VA`, `PU3110VA`, `PU3120VA`, `PU3220VA`, `PU3320VA`) — plusieurs
cycles réels fusionnés en une seule "opération" comptée. Pour chacun,
`pas_attente` du catalogue n'apparaît quasiment jamais dans le signal réel
(ex. `PU3220VA`/`PU3320VA` : jamais du tout, cf. section 3).

Recherche du **vrai** pas de redémarrage par régularité d'espacement : parmi
les pas les plus fréquents d'un tag, un vrai redémarrage revient à
intervalles réguliers (CV = écart-type / moyenne des écarts bas) ; un pas
répété plusieurs fois *par* cycle (ex. oscillation d'un régulateur autour
d'un seuil, cf. `PU3320VA` : pas 11 apparaît deux fois par cycle à
`[...,10,11,12,11,15,...]`) revient de façon erratique (CV élevé) et sur-
compte les opérations s'il est pris comme ancre.

`suspect=True` : le pas déclaré apparaît beaucoup moins souvent que le
candidat le plus régulier trouvé dans les données — le catalogue mérite une
relecture du `OPxxxxVA.DOC` correspondant (cf. `OPERATION.md`, étape 1 :
c'est un jugement métier, pas quelque chose à corriger automatiquement ici).


In [29]:
def anchor_candidates(steps_tag: pd.DataFrame, top_k: int = 6) -> pd.DataFrame:
    """CV (écart-type / moyenne) de l'espacement entre occurrences, pour les
    `top_k` pas les plus fréquents d'un tag. Bas = redémarrage régulier."""
    vc = steps_tag['pas'].value_counts()
    rows = []
    for pas_val in vc.index[:top_k]:
        times = steps_tag.loc[steps_tag['pas'] == pas_val, 'debut'].sort_values()
        gaps = times.diff().dt.total_seconds().dropna() / 60
        if len(gaps) < 3:
            continue
        rows.append({'pas': pas_val, 'n': int(vc[pas_val]), 'cv_espacement': gaps.std() / gaps.mean()})
    return pd.DataFrame(rows).sort_values('cv_espacement')


tag_defs = {t['nom']: t for t in batch_tags}
diagnostic_rows = []
for tag_name, g in pas_tables.groupby('tag'):
    cands = anchor_candidates(g)
    if cands.empty:
        continue
    best = cands.iloc[0]
    declared = tag_defs[tag_name].get('pas_attente')
    n_declared = int((g['pas'] == declared).sum()) if declared else 0
    diagnostic_rows.append({
        'tag': tag_name,
        'pas_attente_catalogue': declared,
        'n_occurrences_declare': n_declared,
        'pas_le_plus_regulier': int(best['pas']),
        'n_occurrences_regulier': int(best['n']),
        'cv_regulier': round(best['cv_espacement'], 2),
    })

diagnostic = pd.DataFrame(diagnostic_rows)
diagnostic['suspect'] = diagnostic['n_occurrences_declare'] < diagnostic['n_occurrences_regulier'] * 0.5
diagnostic.sort_values('n_occurrences_declare')


,tag,pas_attente_catalogue,n_occurrences_declare,pas_le_plus_regulier,n_occurrences_regulier,cv_regulier,suspect
13,PU2810VA,4,619,14,756,1.16,False
2,PU1430VA,13,1375,18,1318,1.81,False
7,PU2310VA,5,3102,14,3095,2.35,False
1,PU1420VA,5,3112,5,3112,2.70,False
9,PU2340VA,4,3128,8,172,1.28,False
8,PU2320VA,4,3144,4,3144,2.05,False
19,PU3310VA,4,3353,6,4010,2.57,False
0,PU1410VA,4,3410,13,3253,2.77,False
3,PU1510VA,3,3533,6,3532,2.69,False
5,PU1530VA,3,3552,4,3212,2.63,False


## 7. Matrice pas × opération

Une ligne par tag, une colonne par pas (1 à 32) : durée **médiane** passée
dans ce pas (minutes), tous cycles confondus sur la période. Vide si ce pas
n'a jamais été observé pour ce tag (pas hors de la plage réelle de
l'opération, cf. `pas_min`/`pas_max` du récap section 5).

Colonnes supplémentaires : `nb_ope`, `temps_reference` (minimum documenté),
`temps_ope_min`/`_median`/`_moyen` (durée totale d'opération, section 4 —
`_moyen` tiré vers le haut par les grandes valeurs extrêmes visibles en
section 4/6, `_median` plus représentatif du cas courant).


In [31]:
matrice = (
    pas_tables[pas_tables['operation'] > 0]
    .pivot_table(index='tag', columns='pas', values='duree_min', aggfunc='median')
    .reindex(columns=range(PAS_MIN, PAS_MAX + 1))
)
matrice.columns = [f'pas_{c}' for c in matrice.columns]
matrice = matrice.round(1)

duree_moyenne = operations.groupby('tag')['duree_min'].mean().rename('duree_minutes_moyenne')
matrice = matrice.join(
    recap.set_index('tag')[['nb_ope', 'temps_reference_min', 'duree_minutes_min', 'duree_minutes_median']]
).join(duree_moyenne)
matrice = matrice.rename(columns={
    'temps_reference_min': 'temps_reference',
    'duree_minutes_min': 'temps_ope_min',
    'duree_minutes_median': 'temps_ope_median',
    'duree_minutes_moyenne': 'temps_ope_moyen',
})
matrice['temps_ope_moyen'] = matrice['temps_ope_moyen'].round(1)

MATRIX_PATH = "documents/confidentiel/matrice_pas_operations.csv"
matrice.to_csv(MATRIX_PATH)
print(f"Matrice sauvegardée : {MATRIX_PATH} ({matrice.shape[0]} opérations × {matrice.shape[1]} colonnes)")
matrice.fillna('')  # affichage seulement -- le CSV ci-dessus reste numérique (cases vides en NaN, jamais le texte "NaN")


Matrice sauvegardée : documents/confidentiel/matrice_pas_operations.csv (21 opérations × 37 colonnes)


,pas_1,pas_2,pas_3,pas_4,pas_5,pas_6,pas_7,pas_8,pas_9,pas_10,pas_11,pas_12,pas_13,pas_14,pas_15,pas_16,pas_17,pas_18,pas_19,pas_20,pas_21,pas_22,pas_23,pas_24,pas_25,pas_26,pas_27,pas_28,pas_29,pas_30,pas_31,pas_32,nb_ope,temps_reference,temps_ope_min,temps_ope_median,temps_ope_moyen
tag,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
PU1410VA,16.0,4.0,1.0,6.0,2.0,13.0,38.0,7.0,1.0,7.0,89.0,28.0,9.0,30.0,23.0,146.0,30.0,8.0,1.0,2.0,49.0,14.0,78.0,,,,,,,,,,3086,425,426.0,541.0,780.6
PU1420VA,119.0,186.0,1.0,13.0,10.0,66.0,225.0,16.0,16.5,1.0,1.0,1.0,1.0,9.0,12.5,,,,,,,,,,,,,,,,,,3096,222,235.0,536.0,778.1
PU1430VA,507.0,4.0,1.0,1.0,17.0,1.0,5.0,1.0,7.0,2.0,27.0,1.0,227.0,12.0,57.0,5.0,448.0,48.0,6.0,51.0,12.0,9.0,1.5,6.0,20.0,15.0,17.0,7.0,,,,,1038,1027,1052.0,1738.0,2319.7
PU1510VA,3.0,16.5,15.0,11.0,61.0,12.0,1.0,306.0,54.0,1.0,,1.0,,1.0,1.0,1.0,,4.0,1.0,1.0,3.0,1.5,1.0,2.0,1.0,1.0,,2.0,,17.0,,,3519,412,418.0,480.0,684.6
PU1520VA,19.5,37.0,32.0,5.0,104.0,127.0,10.0,8.0,135.0,6.0,20.0,8.0,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.5,1.0,2.0,13.0,,,3479,387,387.0,483.0,692.4
PU1530VA,26.0,18.0,39.0,2.0,132.0,100.0,153.0,34.0,,1.0,,1.0,,1.0,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,,1.0,2.0,15.0,,,3392,418,418.0,489.5,710.1
PU1540VA,3.0,31.0,19.0,50.0,8.0,124.0,60.0,4.0,6.0,6.0,65.0,3.0,6.0,65.0,3.0,27.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.5,1.0,1.0,10.0,,,3478,391,391.0,486.0,692.6
PU2310VA,235.0,60.5,1.0,1.0,52.0,63.0,240.0,24.0,9.0,1.0,10.0,45.0,48.0,15.0,5.0,47.0,8.0,,,,,,,,,,,,,,,,3050,530,531.0,578.0,789.7
PU2320VA,47.0,1.0,1.0,109.0,3.0,100.0,51.0,17.0,1.0,7.0,3.5,4.0,21.0,45.0,14.0,6.0,1.0,12.0,145.0,10.0,56.0,8.0,,,,,,,,,,,3097,327,336.0,589.0,777.8
